In [7]:
# Projeto Crypto Analytics

In [8]:
import datetime
import requests
import pandas as pd
import sqlite3

In [9]:
requisicao = requests.get("https://api.coingecko.com/api/v3/simple/price?ids=bitcoin,ethereum,solana,ripples,cardano,binancecoin&vs_currencies=usd,eur,brl&include_market_cap=true&include_24hr_change=true/x-cg-demo-api-key:CG-hhQaHNAtEgaqAsvkPMFWFTXZ")

df_requisicao = pd.DataFrame(requisicao.json())

display(df_requisicao)

,bitcoin,ethereum,solana,ripples,cardano,binancecoin
usd,8.426100e+04,2.671570e+03,1.140700e+02,0.001972,2.367710e-01,7.656100e+02
usd_market_cap,1.693014e+12,3.261932e+11,6.704773e+10,167627.656675,8.890604e+09,1.019744e+11
eur,7.403400e+04,2.347310e+03,1.002200e+02,0.001733,2.080330e-01,6.726900e+02
eur_market_cap,1.487530e+12,2.866025e+11,5.891002e+10,147282.352729,7.811534e+09,8.959756e+10
brl,4.356960e+05,1.381413e+04,5.898300e+02,0.010197,1.220000e+00,3.958820e+03
brl_market_cap,8.754238e+12,1.686680e+12,3.466904e+11,866769.087136,4.597154e+10,5.272892e+11


In [10]:
tickers = {
    "Bitcoin":"BTC",
    "Ethereum":"ETH",
    "Solana":"SOL",
    "Ripples":"XRP",
    "Cardano":"ADA",
    "Binancecoin":"BNB"
}

df_ajustado = df_requisicao.stack(0).unstack(0)
df_ajustado["data_coleta"] = datetime.datetime.now()
df_ajustado = df_ajustado.reset_index(drop=False)
df_ajustado.rename(columns={"index":"coin"}, inplace=True)
df_ajustado["coin"] = df_ajustado["coin"].str.title()
df_ajustado["ticker"] = df_ajustado["coin"].map(tickers)

display(df_ajustado)
df_ajustado.dtypes

,coin,usd,usd_market_cap,eur,eur_market_cap,brl,brl_market_cap,data_coleta,ticker
0,Bitcoin,84261.000000,1.693014e+12,74034.000000,1.487530e+12,435696.000000,8.754238e+12,2026-09-23 17:54:41.166846,BTC
1,Ethereum,2671.570000,3.261932e+11,2347.310000,2.866025e+11,13814.130000,1.686680e+12,2026-09-23 17:54:41.166846,ETH
2,Solana,114.070000,6.704773e+10,100.220000,5.891002e+10,589.830000,3.466904e+11,2026-09-23 17:54:41.166846,SOL
3,Ripples,0.001972,1.676277e+05,0.001733,1.472824e+05,0.010197,8.667691e+05,2026-09-23 17:54:41.166846,XRP
4,Cardano,0.236771,8.890604e+09,0.208033,7.811534e+09,1.220000,4.597154e+10,2026-09-23 17:54:41.166846,ADA
5,Binancecoin,765.610000,1.019744e+11,672.690000,8.959756e+10,3958.820000,5.272892e+11,2026-09-23 17:54:41.166846,BNB


coin                         str
usd                      float64
usd_market_cap           float64
eur                      float64
eur_market_cap           float64
brl                      float64
brl_market_cap           float64
data_coleta       datetime64[us]
ticker                       str
dtype: object

In [11]:
conexao = sqlite3.connect("crypto_analytics.db")

df_ajustado.to_sql(name="tb_crypto_base_api", con=conexao, if_exists="append", index=False)

conexao.close()

In [12]:
conexao = sqlite3.connect("crypto_analytics.db")

resultado = pd.read_sql(
    """
    SELECT * 
    FROM tb_crypto_base_api
    """, 
    conexao
)

display(resultado.head(10))

conexao.close()

,coin,usd,usd_market_cap,eur,eur_market_cap,brl,brl_market_cap,data_coleta,ticker
0,Bitcoin,84226.000000,1.691960e+12,73983.000000,1.486189e+12,435407.000000,8.746589e+12,2026-09-23 15:51:07.406057,BTC
1,Ethereum,2664.700000,3.252895e+11,2340.630000,2.857287e+11,13775.190000,1.681584e+12,2026-09-23 15:51:07.406057,ETH
2,Solana,114.250000,6.713082e+10,100.360000,5.896657e+10,590.620000,3.470328e+11,2026-09-23 15:51:07.406057,SOL
3,Ripples,0.001972,1.676277e+05,0.001732,1.472413e+05,0.010195,8.665512e+05,2026-09-23 15:51:07.406057,XRP
4,Cardano,0.237933,8.929414e+09,0.208997,7.843445e+09,1.230000,4.616061e+10,2026-09-23 15:51:07.406057,ADA
5,Binancecoin,766.760000,1.021021e+11,673.510000,8.968475e+10,3963.780000,5.278168e+11,2026-09-23 15:51:07.406057,BNB
6,Bitcoin,84332.000000,1.692782e+12,74076.000000,1.486911e+12,435957.000000,8.750835e+12,2026-09-23 15:53:19.388226,BTC
7,Ethereum,2668.780000,3.254925e+11,2344.210000,2.859071e+11,13796.240000,1.682633e+12,2026-09-23 15:53:19.388226,ETH
8,Solana,114.440000,6.716808e+10,100.520000,5.899930e+10,591.570000,3.472254e+11,2026-09-23 15:53:19.388226,SOL
9,Ripples,0.001972,1.676277e+05,0.001732,1.472413e+05,0.010195,8.665512e+05,2026-09-23 15:53:19.388226,XRP
